# Table 1 — baseline characteristics and weighting

**Who was compared, in numbers.** One table doing what `T02` and `T05` do separately:
the clinically recognisable baseline by arm, unweighted and overlap-weighted, beside the standardised
mean difference before and after weighting. Centre is a row here like any other covariate, because in
this cohort it behaves like the strongest one.

This notebook **composes**; it computes nothing. Every number comes off the cached run that
`manuscript.load()` returns, every sentence beneath the exhibit comes off `report.py`, and the
artefacts land in `out/manuscript/`. Section references `[§n]` point to `../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all seven exhibits. The record below says what it depends on.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)                # control first

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The rows

`report.baseline_summary` expands every `[§9]` balance covariate over its **declared** levels, so a
level nobody has renders as a zero row rather than vanishing, and summarises each row the way a
clinical table is read: `n (%)` where the covariate is a factor level or a 0/1 flag, `median (IQR)`
otherwise. Which rows are counted is declared in `config.py`, not inferred from what this cohort
happens to contain.

A weighted arm has no integer count — its size is the effective sample size in the last row — so its
count cells carry the share alone. Its `median (IQR)` is the weighted quantile under the same
inverted-CDF rule the `[§10]` percentile intervals use.

The SMD columns come off the balance record, which was computed against the same declared levels.

In [ ]:
rows = report.baseline_summary(bundle.cohort, bundle.ps)
baseline = pd.DataFrame(list(rows[1:]), columns=list(rows[0]))

# Three decimals on an SMD: the threshold it is judged against has two, and T05 carries the rest.
smd = pd.DataFrame([(r.covariate, r.role, f"{r.unweighted:.3f}", f"{r.weighted:.3f}",
                     "yes" if abs(r.weighted) >= C.SMD_THRESHOLD else "no")
                    for r in bundle.balance.covariates],
                   columns=["covariate", "role", "SMD unweighted", "SMD weighted", "exceeds"])

table = baseline.merge(smd, on="covariate", how="left").fillna("—")
table

The effective sample size row is the last one: what the weighted arms are worth in
equally weighted patients. It carries no SMD, which is why the merge leaves it dashed.

In [ ]:
pd.DataFrame({"patients": {ARM[a]: int((bundle.cohort.loc[bundle.ps.in_model, C.TREATMENT] == a).sum())
                          for a in ARMS},
              "effective sample size": {ARM[a]: bundle.ps.ess[a] for a in ARMS}})

## 3. Write it

In [ ]:
path = M.save_table("Table_1", table, (
    "Table 1. Baseline characteristics and weighting diagnostics in the [§7] overlap population. "
    "Counts are n (%) and continuous variables median (IQR), by arm before and after overlap "
    "weighting, with the standardised mean difference on both. A weighted arm has no integer count — "
    "its size is the effective sample size row — so its count cells carry the share alone and its "
    "medians are weighted quantiles. Roles are to the [§7] prespecified specification.",
    report.ATO_DESCRIPTION,
    f"Balance is judged against the full [§6] confounder set at |SMD| < {report.fmt(C.SMD_THRESHOLD)}, "
    "not only the covariates the propensity model used [§9].",
    report.exceedance_clause(bundle.balance),
    M.provenance_clause(bundle)))

print(path)
pd.read_csv(path, comment="#").head()

## 4. Reading it

- **The weighted columns describe the overlap population, not the cohort** — patients whose treatment
  was genuinely in equipoise at these centres `[§7]`. A different propensity model would target a
  different population.
- **Negative controls are in the table and not in the model.** They are there to be looked at: three
  of them are among the covariates still imbalanced after weighting.
- **The effective sample size, not the headcount, is what the intervals reflect.**